# 21 Local Twitter Dataset Profiling

Local-only profiling of the validated Twitter trending snapshot to define exact normalization requirements for Phase 22.


**Notebook purpose:** Deep local-only profiling of the Twitter trending snapshot: missingness, duplicates, noise analysis, temporal coverage, outlier detection, and normalization rule recommendations for Phase 22.

**Required data:** Twitter trending parquet + metadata + validation report + sample files under `local/reference_snapshots/twitter_trending/` and `data/samples/`. Same inputs as notebook 01.

**Run order:** Run after notebook 01/02 (snapshot and initial normalization). Run before notebook 22 (full normalization).

## 1. Load Validated Local Artifacts


In [1]:
from __future__ import annotations

import json
import math
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


def find_repo_root() -> Path:
    probe = Path.cwd().resolve()
    for candidate in [probe, *probe.parents]:
        if (candidate / 'docs' / '21_local_twitter_dataset_profiling.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root with docs/21_local_twitter_dataset_profiling.md')


ROOT = find_repo_root()
FULL_PARQUET_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_full.parquet'
METADATA_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_snapshot_metadata.json'
SAMPLE_PARQUET_PATH = ROOT / 'data/samples/twitter_trending_sample_1000.parquet'
SAMPLE_CSV_PATH = ROOT / 'data/samples/twitter_trending_sample_1000.csv'
VALIDATION_REPORT_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_validation_report.json'
PROFILE_SUMMARY_PATH = ROOT / 'local/reference_snapshots/twitter_trending/twitter_trending_profile_summary.json'

required_paths = [
    FULL_PARQUET_PATH,
    METADATA_PATH,
    SAMPLE_PARQUET_PATH,
    SAMPLE_CSV_PATH,
    VALIDATION_REPORT_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('Missing required local artifacts: ' + ', '.join(missing_paths))

validation_report = json.loads(VALIDATION_REPORT_PATH.read_text(encoding='utf-8'))
if validation_report.get('status') == 'fail' or validation_report.get('overall_pass') is False:
    raise RuntimeError('Validation report indicates fail status; stop profiling phase.')

print('repo_root:', ROOT)
print('validation_status:', validation_report.get('status'))


repo_root: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject
validation_status: pass


In [2]:
df = pd.read_parquet(FULL_PARQUET_PATH)
name_series = pd.Series(['' if v is None else str(v) for v in df['name'].astype('object').tolist()], dtype='object')
date_series = pd.to_datetime(df['date'], errors='coerce')
counts_series = pd.to_numeric(df['counts'], errors='coerce')
num_hours_series = pd.to_numeric(df['num_hours'], errors='coerce')

print('rows:', len(df))
print('columns:', list(df.columns))


rows: 101731
columns: ['num_hours', 'date', 'name', 'counts']


## 2. Dataset Shape And Schema


In [3]:
shape_df = pd.DataFrame([
    {
        'row_count': int(len(df)),
        'column_count': int(len(df.columns)),
        'unique_name_count': int(name_series.nunique(dropna=True)),
        'unique_date_count': int(date_series.dt.date.nunique()),
        'unique_date_name_pairs': int(df[['date', 'name']].drop_duplicates().shape[0]),
    }
])

schema_df = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(dtype) for dtype in df.dtypes],
})

shape_df, schema_df


(   row_count  column_count  unique_name_count  unique_date_count  \
 0     101731             4              33973                749   
 
    unique_date_name_pairs  
 0                  101609  ,
       column    dtype
 0  num_hours     int8
 1       date   object
 2       name      str
 3     counts  float64)

## 3. Missingness (Nulls And Blanks)


In [4]:
null_counts = {col: int(val) for col, val in df.isna().sum().items()}
blank_counts = {}
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) or df[col].dtype == object:
        blank_counts[col] = int(
            df[col]
            .astype('object')
            .map(lambda v: '' if v is None else str(v))
            .str.strip()
            .eq('')
            .sum()
        )

missingness_df = pd.DataFrame({
    'column': list(df.columns),
    'null_count': [null_counts[col] for col in df.columns],
    'blank_count': [blank_counts.get(col, 0) for col in df.columns],
})
missingness_df['null_rate'] = missingness_df['null_count'] / len(df)
missingness_df['blank_rate'] = missingness_df['blank_count'] / len(df)

columns_with_no_missing = missingness_df.loc[missingness_df['null_count'] == 0, 'column'].tolist()
columns_with_partial_missing = missingness_df.loc[
    (missingness_df['null_count'] > 0) & (missingness_df['null_count'] < len(df)),
    'column',
].tolist()
columns_with_severe_missing = missingness_df.loc[missingness_df['null_rate'] >= 0.2, 'column'].tolist()

missingness_df


,column,null_count,blank_count,null_rate,blank_rate
0,num_hours,0,0,0.0,0.0
1,date,0,0,0.0,0.0
2,name,0,0,0.0,0.0
3,counts,0,0,0.0,0.0


## 4. Duplicates (Explicit Definitions)


In [5]:
duplicate_metrics = {
    'full_row_duplicates': int(df.duplicated().sum()),
    'duplicate_date_name_pairs': int(df.duplicated(subset=['date', 'name']).sum()),
    'repeated_names_ge_2': int((name_series.value_counts(dropna=True) >= 2).sum()),
    'repeated_names_ge_10': int((name_series.value_counts(dropna=True) >= 10).sum()),
}

top_repeated_names_df = (
    name_series.value_counts(dropna=True)
    .head(20)
    .rename_axis('name')
    .reset_index(name='count')
)

pd.DataFrame([duplicate_metrics]), top_repeated_names_df


(   full_row_duplicates  duplicate_date_name_pairs  repeated_names_ge_2  \
 0                  122                        122                14119   
 
    repeated_names_ge_10  
 0                  1884  ,
                  name  count
 0             #OPLive    119
 1             #WWERaw    116
 2          #SmackDown    109
 3             #WWENXT    109
 4        Good Tuesday    107
 5         Good Monday    107
 6       Good Saturday    106
 7      #FursuitFriday    106
 8         Good Sunday    106
 9         Good Friday    105
 10  #MondayMotivation    104
 11       #AEWDynamite    103
 12      Good Thursday    103
 13       #FridayVibes    103
 14     Good Wednesday    102
 15          #Caturday    101
 16          Liverpool    100
 17           Hump Day    100
 18       #sundayvibes     97
 19     #SaturdayVibes     92)

## 5. Value Distributions (Text, Date, Numeric)


In [6]:
name_char_len = name_series.str.len()
name_token_len = name_series.str.split().map(len)

distribution_df = pd.DataFrame([
    {
        'counts_min': float(counts_series.min()),
        'counts_p50': float(counts_series.quantile(0.5)),
        'counts_p95': float(counts_series.quantile(0.95)),
        'counts_p99': float(counts_series.quantile(0.99)),
        'counts_max': float(counts_series.max()),
        'num_hours_min': float(num_hours_series.min()),
        'num_hours_p50': float(num_hours_series.quantile(0.5)),
        'num_hours_p95': float(num_hours_series.quantile(0.95)),
        'num_hours_p99': float(num_hours_series.quantile(0.99)),
        'num_hours_max': float(num_hours_series.max()),
        'name_char_len_p50': float(name_char_len.quantile(0.5)),
        'name_char_len_p95': float(name_char_len.quantile(0.95)),
        'name_char_len_max': float(name_char_len.max()),
        'name_token_len_p50': float(name_token_len.quantile(0.5)),
        'name_token_len_p95': float(name_token_len.quantile(0.95)),
        'name_token_len_max': float(name_token_len.max()),
    }
])

distribution_df


,counts_min,counts_p50,counts_p95,counts_p99,counts_max,num_hours_min,num_hours_p50,num_hours_p95,num_hours_p99,num_hours_max,name_char_len_p50,name_char_len_p95,name_char_len_max,name_token_len_p50,name_token_len_p95,name_token_len_max
0,0.0,14481.0,253503.0,859933.7,10652034.0,1.0,2.0,9.0,13.0,24.0,8.0,18.0,30.0,1.0,2.0,7.0


## 6. Trend Naming Noise And Normalization Needs


In [7]:
hashtag_pattern = re.compile(r'#[A-Za-z0-9_]+')
url_pattern = re.compile(r'https?://|www\.|\b[a-z0-9-]+\.[a-z]{2,}\b', re.IGNORECASE)
emoji_pattern = re.compile(r'[\U0001F300-\U0001FAFF\U00002600-\U000027BF]')
punct_pattern = re.compile(r'[^A-Za-z0-9#\s]')
non_ascii_pattern = re.compile(r'[^\x00-\x7F]')
multisep_pattern = re.compile(r'[-_./]{2,}|\s{2,}')

hashtags = []
url_rows = emoji_rows = non_ascii_rows = punctuation_rows = 0
leading_trailing_ws_rows = repeated_separator_rows = 0
very_short_rows = low_information_rows = stopword_heavy_rows = 0
hashtag_rows = 0
stopwords = {'the', 'a', 'an', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'at', 'by', 'with', 'from'}

for value in name_series.tolist():
    stripped = value.strip()
    if stripped.startswith('#'):
        hashtag_rows += 1
    hashtags.extend(hashtag_pattern.findall(value))
    if url_pattern.search(value):
        url_rows += 1
    if emoji_pattern.search(value):
        emoji_rows += 1
    if non_ascii_pattern.search(value):
        non_ascii_rows += 1
    if punct_pattern.search(value):
        punctuation_rows += 1
    if re.match(r'^\s|\s$', value):
        leading_trailing_ws_rows += 1
    if multisep_pattern.search(value):
        repeated_separator_rows += 1
    if len(stripped) <= 3:
        very_short_rows += 1

    tokens = [tok for tok in re.split(r'\s+', stripped) if tok]
    if len(tokens) <= 1 and len(stripped) <= 4:
        low_information_rows += 1
    if tokens:
        stop_count = sum(1 for tok in tokens if tok.lower() in stopwords)
        if len(tokens) >= 2 and (stop_count / len(tokens)) >= 0.6:
            stopword_heavy_rows += 1

case_variant_collisions = int(name_series.groupby(name_series.str.lower()).nunique().gt(1).sum())

simple_key = (
    name_series
    .str.normalize('NFKC')
    .str.lower()
    .str.replace('’', "'", regex=False)
    .str.replace(r'[^a-z0-9#\s]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

lexical_collision_counts = simple_key.value_counts()
lexical_collision_keys = lexical_collision_counts[lexical_collision_counts >= 2]
lexical_collision_examples = []
for key in lexical_collision_keys.head(20).index:
    variants = name_series[simple_key == key].drop_duplicates().head(5).tolist()
    if len(variants) > 1:
        lexical_collision_examples.append({'normalized_key': key, 'variants': variants})

noise_summary_df = pd.DataFrame([
    {
        'case_variant_collisions': int(case_variant_collisions),
        'hashtag_rows': int(hashtag_rows),
        'unique_hashtags': int(pd.Series(hashtags, dtype='object').nunique()) if hashtags else 0,
        'url_like_rows': int(url_rows),
        'emoji_rows': int(emoji_rows),
        'non_ascii_rows': int(non_ascii_rows),
        'punctuation_rows': int(punctuation_rows),
        'leading_or_trailing_whitespace_rows': int(leading_trailing_ws_rows),
        'repeated_separator_rows': int(repeated_separator_rows),
        'very_short_name_rows_len_le_3': int(very_short_rows),
        'low_information_rows': int(low_information_rows),
        'stopword_heavy_rows': int(stopword_heavy_rows),
        'lexical_collision_key_count_ge_2': int(lexical_collision_keys.shape[0]),
    }
])

top_hashtags_df = (
    pd.Series(hashtags, dtype='object').value_counts().head(20).rename_axis('hashtag').reset_index(name='count')
    if hashtags else pd.DataFrame(columns=['hashtag', 'count'])
)

lexical_collision_examples_df = pd.DataFrame(lexical_collision_examples[:10])
noise_summary_df, top_hashtags_df, lexical_collision_examples_df


(   case_variant_collisions  hashtag_rows  unique_hashtags  url_like_rows  \
 0                     1281         17180             7032              0   
 
    emoji_rows  non_ascii_rows  punctuation_rows  \
 0           0             546              4406   
 
    leading_or_trailing_whitespace_rows  repeated_separator_rows  \
 0                                    0                        2   
 
    very_short_name_rows_len_le_3  low_information_rows  stopword_heavy_rows  \
 0                              0                  8327                   33   
 
    lexical_collision_key_count_ge_2  
 0                             14178  ,
                  hashtag  count
 0                #OPLive    119
 1                #WWERaw    116
 2             #SmackDown    109
 3                #WWENXT    109
 4         #FursuitFriday    106
 5      #MondayMotivation    104
 6           #AEWDynamite    103
 7           #FridayVibes    103
 8              #Caturday    101
 9           #sundayvibes    

## 7. Temporal Coverage And Daily Density


In [8]:
rows_per_day = date_series.dropna().dt.date.value_counts().sort_index()

if len(rows_per_day) > 0:
    date_min = str(rows_per_day.index.min())
    date_max = str(rows_per_day.index.max())
    full_range = pd.date_range(rows_per_day.index.min(), rows_per_day.index.max(), freq='D').date
    missing_dates_in_range = int(len(set(full_range) - set(rows_per_day.index)))
else:
    date_min = None
    date_max = None
    missing_dates_in_range = 0

temporal_summary_df = pd.DataFrame([
    {
        'date_min': date_min,
        'date_max': date_max,
        'unique_dates': int(rows_per_day.shape[0]),
        'missing_dates_in_min_max_range': missing_dates_in_range,
        'daily_rows_min': int(rows_per_day.min()) if len(rows_per_day) else 0,
        'daily_rows_max': int(rows_per_day.max()) if len(rows_per_day) else 0,
        'daily_rows_mean': float(rows_per_day.mean()) if len(rows_per_day) else 0.0,
        'daily_rows_median': float(rows_per_day.median()) if len(rows_per_day) else 0.0,
        'sparse_days_lt_50_rows': int((rows_per_day < 50).sum()) if len(rows_per_day) else 0,
        'sparse_days_lt_100_rows': int((rows_per_day < 100).sum()) if len(rows_per_day) else 0,
    }
])

rows_per_day_df = rows_per_day.rename_axis('date').reset_index(name='row_count')
rows_per_day_df['date'] = rows_per_day_df['date'].astype(str)

temporal_summary_df, rows_per_day_df.head(10), rows_per_day_df.tail(10)


(     date_min    date_max  unique_dates  missing_dates_in_min_max_range  \
 0  2024-01-01  2026-01-24           749                               6   
 
    daily_rows_min  daily_rows_max  daily_rows_mean  daily_rows_median  \
 0              35             244        135.82243              136.0   
 
    sparse_days_lt_50_rows  sparse_days_lt_100_rows  
 0                       4                       11  ,
          date  row_count
 0  2024-01-01        102
 1  2024-01-02        122
 2  2024-01-03        104
 3  2024-01-04        110
 4  2024-01-05        118
 5  2024-01-06        115
 6  2024-01-07        122
 7  2024-01-08        119
 8  2024-01-09        128
 9  2024-01-10        130,
            date  row_count
 739  2026-01-15         67
 740  2026-01-16        134
 741  2026-01-17        170
 742  2026-01-18        149
 743  2026-01-19        124
 744  2026-01-20        129
 745  2026-01-21        154
 746  2026-01-22        148
 747  2026-01-23        147
 748  2026-01-24    

## 8. Suspicious Numeric Values And Outliers


In [9]:
def iqr_outlier_stats(series: pd.Series) -> dict[str, float | int | None]:
    clean = pd.to_numeric(series, errors='coerce').dropna()
    if clean.empty:
        return {'row_count': 0, 'iqr_lower': None, 'iqr_upper': None, 'outlier_count': 0, 'outlier_ratio': 0.0}
    q1 = float(clean.quantile(0.25))
    q3 = float(clean.quantile(0.75))
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = int(((clean < lower) | (clean > upper)).sum())
    return {
        'row_count': int(len(clean)),
        'iqr_lower': lower,
        'iqr_upper': upper,
        'outlier_count': outliers,
        'outlier_ratio': float(outliers / len(clean)),
    }

numeric_flags_df = pd.DataFrame([
    {
        'counts_zero_rows': int((counts_series == 0).sum()),
        'counts_negative_rows': int((counts_series < 0).sum()),
        'num_hours_zero_rows': int((num_hours_series == 0).sum()),
        'num_hours_negative_rows': int((num_hours_series < 0).sum()),
        'num_hours_gt_24_rows': int((num_hours_series > 24).sum()),
    }
])

counts_outliers_df = pd.DataFrame([iqr_outlier_stats(counts_series)])
num_hours_outliers_df = pd.DataFrame([iqr_outlier_stats(num_hours_series)])

numeric_flags_df, counts_outliers_df, num_hours_outliers_df


(   counts_zero_rows  counts_negative_rows  num_hours_zero_rows  \
 0              1395                     0                    0   
 
    num_hours_negative_rows  num_hours_gt_24_rows  
 0                        0                     0  ,
    row_count  iqr_lower  iqr_upper  outlier_count  outlier_ratio
 0     101731  -56689.25  107364.75          11999       0.117948,
    row_count  iqr_lower  iqr_upper  outlier_count  outlier_ratio
 0     101731       -5.0       11.0           2202       0.021645)

## 9. Recommended Normalization Rules (Phase 22)


In [10]:
recommended_rules = [
    'Preserve raw columns (`name`, `date`, `num_hours`, `counts`) unchanged and add derived normalized fields.',
    'Unicode-normalize trend names with NFKC, then normalize curly apostrophes (`’`/`‘`) to ASCII apostrophe.',
    'Trim leading/trailing whitespace and collapse repeated internal whitespace to single spaces.',
    'Case-fold to lowercase for canonical matching keys while keeping raw display text separately.',
    'Create dual hashtag-aware keys: one preserving leading `#` and one with a single leading `#` removed.',
    'Normalize separator punctuation (`&`, `-`, `.`, `,`, `/`, `_`) to spaces before tokenization.',
    'Create a ticker-aware variant by removing a single leading `$` for symbol-prefixed topics.',
    'Create an alphanumeric helper key by stripping non-alphanumeric characters after normalization.',
    'Add helper flags and metrics (`is_hashtag`, `has_special_chars`, `has_non_ascii`, token count, char count).',
    'Deduplicate prepared trend rows by (`date`, normalized_no_hash_key`) in Phase 22.',
]

for i, rule in enumerate(recommended_rules, start=1):
    print(f'{i}. {rule}')


1. Preserve raw columns (`name`, `date`, `num_hours`, `counts`) unchanged and add derived normalized fields.
2. Unicode-normalize trend names with NFKC, then normalize curly apostrophes (`’`/`‘`) to ASCII apostrophe.
3. Trim leading/trailing whitespace and collapse repeated internal whitespace to single spaces.
4. Case-fold to lowercase for canonical matching keys while keeping raw display text separately.
5. Create dual hashtag-aware keys: one preserving leading `#` and one with a single leading `#` removed.
6. Normalize separator punctuation (`&`, `-`, `.`, `,`, `/`, `_`) to spaces before tokenization.
7. Create a ticker-aware variant by removing a single leading `$` for symbol-prefixed topics.
8. Create an alphanumeric helper key by stripping non-alphanumeric characters after normalization.
9. Add helper flags and metrics (`is_hashtag`, `has_special_chars`, `has_non_ascii`, token count, char count).
10. Deduplicate prepared trend rows by (`date`, normalized_no_hash_key`) in Phase 22

In [11]:
profile_summary = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'validation_gate': {
        'status': validation_report.get('status'),
        'overall_pass': validation_report.get('overall_pass'),
        'warnings': validation_report.get('warnings', []),
        'failures': validation_report.get('failures', []),
    },
    'shape_schema': {
        'row_count': int(len(df)),
        'column_count': int(len(df.columns)),
        'column_names': list(df.columns),
        'dtypes': {col: str(dtype) for col, dtype in df.dtypes.items()},
        'candidate_uniqueness': {
            'unique_name_count': int(name_series.nunique(dropna=True)),
            'unique_date_count': int(date_series.dt.date.nunique()),
            'unique_date_name_pairs': int(df[['date', 'name']].drop_duplicates().shape[0]),
        },
    },
    'missingness': {
        'null_counts_by_column': null_counts,
        'blank_string_counts_by_column': blank_counts,
        'columns_with_no_missingness': columns_with_no_missing,
        'columns_with_partial_missingness': columns_with_partial_missing,
        'columns_with_severe_missingness': columns_with_severe_missing,
    },
    'duplicates': {
        'definitions_checked': ['full_row_duplicates', 'duplicate_date_name_pairs', 'repeated_names_across_dates'],
        'full_row_duplicates': duplicate_metrics['full_row_duplicates'],
        'duplicate_date_name_pairs': duplicate_metrics['duplicate_date_name_pairs'],
        'repeated_name_count_ge_2': duplicate_metrics['repeated_names_ge_2'],
        'repeated_name_count_ge_10': duplicate_metrics['repeated_names_ge_10'],
        'top_repeated_names': top_repeated_names_df.to_dict(orient='records'),
    },
    'value_distributions': distribution_df.to_dict(orient='records')[0],
    'naming_noise': {
        **noise_summary_df.to_dict(orient='records')[0],
        'top_hashtags': top_hashtags_df.to_dict(orient='records'),
        'lexical_collision_examples': lexical_collision_examples_df.to_dict(orient='records'),
    },
    'temporal': {
        **temporal_summary_df.to_dict(orient='records')[0],
        'daily_row_count_head': rows_per_day_df.head(10).to_dict(orient='records'),
        'daily_row_count_tail': rows_per_day_df.tail(10).to_dict(orient='records'),
    },
    'numeric_suspicious': {
        **numeric_flags_df.to_dict(orient='records')[0],
        'counts_iqr_outliers': counts_outliers_df.to_dict(orient='records')[0],
        'num_hours_iqr_outliers': num_hours_outliers_df.to_dict(orient='records')[0],
    },
    'recommended_normalization_rules_phase_22': recommended_rules,
    'go_no_go_phase_22': {
        'ready_for_phase_22_normalization': True,
        'reason': 'Profiling is complete, data quality is understood, and deterministic normalization rules are specified.',
    },
}

PROFILE_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
PROFILE_SUMMARY_PATH.write_text(json.dumps(profile_summary, indent=2), encoding='utf-8')
print('wrote profile summary:', PROFILE_SUMMARY_PATH)


wrote profile summary: /Users/quintonevans/Desktop/Neumont/DataEngineeringProjectClass/AppliedDataEngineeringBlueSkyProject/local/reference_snapshots/twitter_trending/twitter_trending_profile_summary.json


In [12]:
phase_22_ready = bool(profile_summary['go_no_go_phase_22']['ready_for_phase_22_normalization'])
print('phase_22_ready:', phase_22_ready)
print('go_no_go_reason:', profile_summary['go_no_go_phase_22']['reason'])


phase_22_ready: True
go_no_go_reason: Profiling is complete, data quality is understood, and deterministic normalization rules are specified.
